# Matrix Factorizations & SVD

矩阵分解与奇异值分解。从 LU、QR、Cholesky 三大分解到 SVD，从低秩近似到图像压缩，全程配代码与可视化。

> 本节课代码为主。SVD 是核心，第 5-8 节要连起来看。

## 0. 环境配置与导入

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 为什么需要矩阵分解？

矩阵分解（Matrix Factorization）是将一个矩阵分解为两个或多个简单矩阵的乘积。

**为什么不直接用原始矩阵？**

| 问题 | 直接做法 | 分解后做法 | 优势 |
|------|---------|-----------|------|
| 解 $Ax=b$ | $x=A^{-1}b$（$O(n^3)$，不稳定） | LU 分解后回代（$O(n^2)$ 每次） | 多次求解时快得多 |
| 求正交基 | 反复归一化 | QR 分解 | 数值稳定 |
| 求逆/行列式 | 直接计算 | 三角分解 | 稳定高效 |
| 降维/压缩 | 保留全部 | 截断 SVD | 保留最重要信息 |
| 求伪逆 | $(A^TA)^{-1}A^T$ | SVD 伪逆 | 适用于奇异矩阵 |

**本节课覆盖的四大分解**：
1. **LU 分解**：$PA = LU$（高斯消元的矩阵形式）
2. **QR 分解**：$A = QR$（正交矩阵 × 上三角矩阵）
3. **Cholesky 分解**：$A = LL^T$（正定矩阵专属）
4. **SVD**：$A = U\Sigma V^T$（最通用的分解，适用于任意矩阵）

In [ ]:
# 一个直观例子：解线性方程组
# 直接求逆 vs LU分解（概念对比，后续章节详细展开）
A = torch.randn(100, 100)
b = torch.randn(100)

import time

# 方法1：直接求逆
start = time.time()
for _ in range(100):
    x1 = torch.linalg.inv(A) @ b
time_inv = (time.time() - start) / 100

# 方法2：linalg.solve（内部用LU分解）
start = time.time()
for _ in range(100):
    x2 = torch.linalg.solve(A, b)
time_solve = (time.time() - start) / 100

print(f"100x100 矩阵解 Ax=b:")
print(f"  求逆法耗时: {time_inv*1000:.3f} ms")
print(f"  solve(LU) 耗时: {time_solve*1000:.3f} ms")
print(f"  结果一致? {torch.allclose(x1, x2, atol=1e-4)}")
print(f"  solve 更快: {time_inv/time_solve:.1f}x")
print("\n→ 这就是矩阵分解的价值：一次分解，多次高效使用")

## 2. LU 分解

### 2.1 定义

LU 分解将矩阵 A 分解为**下三角矩阵 L** 和**上三角矩阵 U** 的乘积：

$$
A = LU
$$

其中 L 是单位下三角矩阵（对角线全为 1），U 是上三角矩阵。

实际中为了数值稳定，需要**选主元**（pivoting），因此更一般的形式是：

$$
PA = LU
$$

其中 P 是置换矩阵（permutation matrix），用于行交换。

**几何意义**：LU 分解是高斯消元法的矩阵形式。L 记录消元过程中的行变换系数，U 是消元后的上三角矩阵。

In [ ]:
A = torch.tensor([[2.0, 1.0, 1.0],
                  [4.0, 3.0, 3.0],
                  [8.0, 7.0, 9.0]])

# torch.linalg.lu 返回 P, L, U（注意 PyTorch 的 P 是置换索引或置换矩阵）
P, L, U = torch.linalg.lu(A)

print("A =\n", A)
print("\nP (置换矩阵) =\n", P)
print("\nL (单位下三角) =\n", L)
print("\nU (上三角) =\n", U)

# 验证 PA = LU
PA = P @ A
LU = L @ U
print("\nP @ A =\n", PA)
print("\nL @ U =\n", LU)
print("\nPA == LU?", torch.allclose(PA, LU, atol=1e-5))

### 2.2 用 LU 分解解线性方程组

对于 $Ax = b$：
1. $PA = LU$（一次分解）
2. 解 $Ly = Pb$（前向代入，$O(n^2)$）
3. 解 $Ux = y$（后向代入，$O(n^2)$）

一旦分解完成，每次求解只需 $O(n^2)$，而直接求逆需要 $O(n^3)$。

In [ ]:
A = torch.tensor([[2.0, 1.0, 1.0],
                  [4.0, 3.0, 3.0],
                  [8.0, 7.0, 9.0]])
b = torch.tensor([4.0, 10.0, 24.0])

# 方法1：直接 solve
x_direct = torch.linalg.solve(A, b)
print("直接 solve: x =", x_direct.tolist())

# 方法2：手动 LU 分解 + 前向/后向代入
P, L, U = torch.linalg.lu(A)

# Step 1: Pb = P @ b
Pb = P @ b
print("\nPb =", Pb.tolist())

# Step 2: 前向代入解 Ly = Pb（L 是单位下三角）
y = torch.zeros_like(Pb)
for i in range(3):
    y[i] = Pb[i] - torch.dot(L[i, :i], y[:i])
print("y (Ly=Pb) =", y.tolist())

# Step 3: 后向代入解 Ux = y（U 是上三角）
x = torch.zeros_like(y)
for i in range(2, -1, -1):
    x[i] = (y[i] - torch.dot(U[i, i+1:], x[i+1:])) / U[i, i]
print("x (Ux=y) =", x.tolist())

print("\n与直接 solve 一致?", torch.allclose(x, x_direct, atol=1e-5))
print("验证 A @ x =", (A @ x).tolist(), "== b?", torch.allclose(A @ x, b))

In [ ]:
# 多次求解的性能优势：一次分解，多次使用
A = torch.randn(200, 200)
n_rhs = 50  # 50 个不同的 b
B = torch.randn(200, n_rhs)

import time

# 方法1：每次都求逆
start = time.time()
X_inv = torch.linalg.inv(A) @ B
time_inv = time.time() - start

# 方法2：LU 分解一次，然后 solve（内部复用分解）
start = time.time()
X_lu = torch.linalg.solve(A, B)
time_lu = time.time() - start

print(f"200x200 矩阵，{n_rhs} 个右端项:")
print(f"  求逆法: {time_inv*1000:.2f} ms")
print(f"  LU分解: {time_lu*1000:.2f} ms")
print(f"  结果一致? {torch.allclose(X_inv, X_lu, atol=1e-3)}")
print(f"  LU 更快: {time_inv/time_lu:.1f}x")

## 3. QR 分解

### 3.1 定义

QR 分解将矩阵 A 分解为**正交矩阵 Q** 和**上三角矩阵 R** 的乘积：

$$
A = QR
$$

其中：
- $Q$ 是正交矩阵：$Q^T Q = I$（列向量两两正交且单位长度）
- $R$ 是上三角矩阵

**几何意义**：QR 分解是 Gram-Schmidt 正交化过程的矩阵形式。Q 的列是 A 的列空间的一组正交基，R 记录从 A 的列到 Q 的列的变换系数。

**应用**：最小二乘法、特征值算法（QR 算法）、正交基构造。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [1.0, 3.0],
                  [1.0, 4.0]])  # 3x2 矩阵

Q, R = torch.linalg.qr(A)

print("A (3x2) =\n", A)
print("\nQ (3x2, 正交列) =\n", Q)
print("\nR (2x2, 上三角) =\n", R)

# 验证 A = Q @ R
print("\nQ @ R =\n", Q @ R)
print("A == Q@R?", torch.allclose(Q @ R, A, atol=1e-5))

# 验证 Q 的列正交：Q^T @ Q = I
print("\nQ^T @ Q =\n", Q.T @ Q)
print("Q^T Q == I?", torch.allclose(Q.T @ Q, torch.eye(2), atol=1e-5))

### 3.2 QR 分解与 Gram-Schmidt 正交化

QR 分解中的 Q 可以通过 Gram-Schmidt 过程构造：对 A 的列向量逐一正交化。

下面手动实现 Gram-Schmidt，验证其结果与 `torch.linalg.qr` 一致。

In [ ]:
def gram_schmidt(A):
    """手动实现 Gram-Schmidt 正交化，返回 Q 和 R"""
    n_rows, n_cols = A.shape
    Q = torch.zeros(n_rows, n_cols)
    R = torch.zeros(n_cols, n_cols)
    
    for j in range(n_cols):
        v = A[:, j].clone()
        # 减去在之前所有正交向量上的投影
        for i in range(j):
            R[i, j] = torch.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]
        # 归一化
        R[j, j] = torch.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    
    return Q, R

A = torch.tensor([[1.0, 2.0],
                  [1.0, 3.0],
                  [1.0, 4.0]])

Q_gs, R_gs = gram_schmidt(A)
Q_torch, R_torch = torch.linalg.qr(A)

print("手动 Gram-Schmidt:")
print("Q =\n", Q_gs)
print("R =\n", R_gs)
print("\nPyTorch QR:")
print("Q =\n", Q_torch)
print("R =\n", R_torch)

# 注意：QR 分解的符号不唯一（Q 的列和 R 的行可以同时变号）
# 所以比较时要看 A = Q@R 是否成立
print("\n手动 GS: A == Q@R?", torch.allclose(Q_gs @ R_gs, A, atol=1e-5))
print("PyTorch: A == Q@R?", torch.allclose(Q_torch @ R_torch, A, atol=1e-5))
print("Q 列正交?", torch.allclose(Q_gs.T @ Q_gs, torch.eye(2), atol=1e-5))

In [ ]:
# QR 分解的应用：求正交基
# 给定一组线性无关的向量，QR 分解给出其张成空间的一组标准正交基
vectors = torch.tensor([[1.0, 1.0, 1.0],
                         [1.0, 2.0, 3.0],
                         [1.0, 4.0, 9.0]]).T  # 3 个 3D 向量作为列

Q, R = torch.linalg.qr(vectors)
print("原始向量（列）:\n", vectors)
print("\n标准正交基 Q（列）:\n", Q)
print("\n验证正交性:")
for i in range(3):
    for j in range(i+1, 3):
        print(f"  q{i+1} · q{j+1} = {torch.dot(Q[:,i], Q[:,j]).item():.6f} (≈0)")
print("\n验证单位长度:")
for i in range(3):
    print(f"  ||q{i+1}|| = {torch.linalg.norm(Q[:,i]).item():.6f} (=1)")

## 4. Cholesky 分解

### 4.1 定义

Cholesky 分解将**正定对称矩阵** A 分解为：

$$
A = LL^T
$$

其中 L 是下三角矩阵，对角线元素为正。

**适用条件**：A 必须是**对称正定矩阵**（symmetric positive definite）。
- 对称：$A = A^T$
- 正定：对任意非零向量 $x$，$x^T A x > 0$
- 等价条件：所有特征值 > 0，所有顺序主子式 > 0

**优势**：Cholesky 分解比 LU 分解快约 2 倍（因为利用了对称性），在优化、统计（协方差矩阵）中广泛使用。

In [ ]:
# 构造一个对称正定矩阵
# 方法：A = B @ B.T + λI（保证正定）
B = torch.randn(3, 3)
A = B @ B.T + 0.5 * torch.eye(3)  # 加单位矩阵保证正定

print("A =\n", A)
print("\nA 对称?", torch.allclose(A, A.T, atol=1e-5))

# 验证正定：所有特征值 > 0
eigenvalues = torch.linalg.eigvalsh(A)
print("\nA 的特征值:", eigenvalues.tolist())
print("所有特征值 > 0?", (eigenvalues > 0).all().item())

# Cholesky 分解
L = torch.linalg.cholesky(A)
print("\nL (下三角) =\n", L)
print("\nL @ L.T =\n", L @ L.T)
print("A == L@L.T?", torch.allclose(L @ L.T, A, atol=1e-5))

### 4.2 Cholesky 分解解线性方程组

对于对称正定系统 $Ax = b$：
1. $A = LL^T$（一次分解）
2. 解 $Ly = b$（前向代入）
3. 解 $L^T x = y$（后向代入）

比 LU 分解更快，因为只需要处理半个矩阵。

In [ ]:
# 对称正定系统
A = torch.tensor([[4.0, 2.0, 1.0],
                  [2.0, 5.0, 2.0],
                  [1.0, 2.0, 6.0]])  # 对称正定
b = torch.tensor([7.0, 9.0, 9.0])

print("A 对称?", torch.allclose(A, A.T))
print("A 特征值:", torch.linalg.eigvalsh(A).tolist(), "(全正 → 正定)")

# Cholesky 分解
L = torch.linalg.cholesky(A)
print("\nL =\n", L)

# 前向代入解 Ly = b
y = torch.zeros(3)
for i in range(3):
    y[i] = (b[i] - torch.dot(L[i, :i], y[:i])) / L[i, i]
print("\ny (Ly=b) =", y.tolist())

# 后向代入解 L^T x = y
x = torch.zeros(3)
LT = L.T
for i in range(2, -1, -1):
    x[i] = (y[i] - torch.dot(LT[i, i+1:], x[i+1:])) / LT[i, i]
print("x (L^T x=y) =", x.tolist())

# 验证
x_direct = torch.linalg.solve(A, b)
print("\n直接 solve: x =", x_direct.tolist())
print("一致?", torch.allclose(x, x_direct, atol=1e-5))
print("A @ x =", (A @ x).tolist(), "== b?", torch.allclose(A @ x, b))

In [ ]:
# 非正定矩阵的 Cholesky 分解会报错
A_singular = torch.tensor([[1.0, 2.0],
                            [2.0, 4.0]])  # det=0，半正定，不是正定
print("奇异矩阵特征值:", torch.linalg.eigvalsh(A_singular).tolist())
try:
    torch.linalg.cholesky(A_singular)
except RuntimeError as e:
    print("Cholesky 报错:", e)

# 负定矩阵也会报错
A_neg_def = torch.tensor([[-1.0, 0.0],
                           [0.0, -2.0]])  # 特征值全负
try:
    torch.linalg.cholesky(A_neg_def)
except RuntimeError as e:
    print("负定矩阵 Cholesky 报错:", e)

print("\n→ Cholesky 分解要求矩阵必须是对称正定的")

## 5. SVD 奇异值分解

### 5.1 定义

**奇异值分解（Singular Value Decomposition, SVD）**是最通用的矩阵分解，适用于**任意形状**的矩阵（不需要方阵，不需要可逆，不需要对称）。

对于任意矩阵 $A \in \mathbb{R}^{m \times n}$，存在分解：

$$
A = U \Sigma V^T
$$

其中：
- $U \in \mathbb{R}^{m \times m}$：正交矩阵，列称为**左奇异向量**
- $\Sigma \in \mathbb{R}^{m \times n}$：对角矩阵（非方阵也可以是"对角"的），对角线元素 $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$ 称为**奇异值**
- $V \in \mathbb{R}^{n \times n}$：正交矩阵，列称为**右奇异向量**
- $r = \text{rank}(A)$：非零奇异值的个数

**SVD 是特征分解的推广**：特征分解只适用于可对角化的方阵，SVD 适用于任意矩阵。

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])  # 2x3 非方阵

# torch.linalg.svd 返回 U, S, Vh（Vh = V^T）
U, S, Vh = torch.linalg.svd(A)

print("A (2x3) =\n", A)
print("\nU (2x2, 左奇异向量) =\n", U)
print("\nS (奇异值，降序) =", S.tolist())
print("\nVh (3x3, V^T, 右奇异向量的转置) =\n", Vh)

# 验证正交性
print("\nU^T @ U =\n", U.T @ U)
print("Vh @ Vh^T =\n", Vh @ Vh.T)

# 重构 A = U @ diag(S) @ Vh
# 注意：S 是 1D，需要构造对角矩阵，且要处理非方阵的情况
Sigma = torch.zeros(A.shape)
for i in range(len(S)):
    Sigma[i, i] = S[i]
A_reconstructed = U @ Sigma @ Vh
print("\n重构 A = U @ Σ @ V^T =\n", A_reconstructed)
print("与原 A 相等?", torch.allclose(A_reconstructed, A, atol=1e-5))

### 5.2 SVD 的几何直觉

SVD 可以理解为线性变换的三步分解：

$$
A = U \Sigma V^T
$$

对任意向量 $x$，$Ax = U(\Sigma(V^T x))$：
1. **$V^T$**：将 $x$ 旋转/反射到右奇异向量坐标系（正交变换，不改变长度）
2. **$\Sigma$**：在各坐标轴方向上按奇异值缩放（可能降维）
3. **$U$**：将缩放后的向量旋转/反射到左奇异向量坐标系

**核心洞察**：任何线性变换都可以分解为"旋转 → 缩放 → 旋转"。奇异值就是各个方向上的缩放因子。

In [ ]:
%matplotlib inline

def visualize_svd_geometry(A, title):
    """可视化 SVD 的三步分解：旋转 → 缩放 → 旋转"""
    U, S, Vh = torch.linalg.svd(A)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # 生成单位圆上的点
    theta = torch.linspace(0, 2 * 3.14159, 200)
    circle = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    
    # Step 0: 原始单位圆
    axes[0].plot(circle[:, 0], circle[:, 1], 'b-')
    axes[0].fill(circle[:, 0], circle[:, 1], 'b', alpha=0.1)
    axes[0].set_title("原始: 单位圆")
    
    # Step 1: V^T 旋转
    step1 = (Vh @ circle.T).T
    axes[1].plot(step1[:, 0], step1[:, 1], 'g-')
    axes[1].fill(step1[:, 0], step1[:, 1], 'g', alpha=0.1)
    axes[1].set_title(f"Step 1: V^T (旋转)")
    
    # Step 2: Σ 缩放
    step2 = step1.clone()
    step2[:, 0] *= S[0]
    step2[:, 1] *= S[1] if len(S) > 1 else 0
    axes[2].plot(step2[:, 0], step2[:, 1], 'orange')
    axes[2].fill(step2[:, 0], step2[:, 1], 'orange', alpha=0.1)
    axes[2].set_title(f"Step 2: Σ (缩放 σ={S[0]:.2f}, {S[1]:.2f})")
    
    # Step 3: U 旋转
    step3 = (U @ step2.T).T
    axes[3].plot(step3[:, 0], step3[:, 1], 'r-')
    axes[3].fill(step3[:, 0], step3[:, 1], 'r', alpha=0.1)
    axes[3].set_title("Step 3: U (旋转) = A")
    
    for ax in axes:
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# 例子1：拉伸变换
A1 = torch.tensor([[3.0, 0.0], [0.0, 1.0]])
visualize_svd_geometry(A1, "拉伸变换 A=[[3,0],[0,1]]")

# 例子2：旋转+拉伸
import math
theta = math.pi / 4
R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta), math.cos(theta)]])
A2 = R @ torch.tensor([[2.0, 0.0], [0.0, 0.5]])
visualize_svd_geometry(A2, "旋转45°+拉伸 A=R·[[2,0],[0,0.5]]")

## 6. SVD 与特征分解的关系

SVD 与特征分解有深刻的联系。对于任意矩阵 $A$：

1. **$A^T A$ 的特征值 = 奇异值的平方**：$\lambda_i(A^T A) = \sigma_i^2$
2. **$A^T A$ 的特征向量 = 右奇异向量**（V 的列）
3. **$AA^T$ 的特征值 = 奇异值的平方**
4. **$AA^T$ 的特征向量 = 左奇异向量**（U 的列）

**证明思路**：
$$
A^T A = (U\Sigma V^T)^T (U\Sigma V^T) = V\Sigma^T U^T U\Sigma V^T = V\Sigma^2 V^T
$$

这正是 $A^T A$ 的特征分解（因为 $A^T A$ 是对称半正定矩阵，可以正交对角化）。

**为什么这很重要**：
- SVD 可以通过特征分解实现（实际算法更复杂，但理论上等价）
- 奇异值一定是非负的（因为是特征值的平方根）
- 非零奇异值的个数 = 矩阵的秩

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

# SVD
U, S, Vh = torch.linalg.svd(A)
print("A 的奇异值 S:", S.tolist())

# A^T A 的特征分解
ATA = A.T @ A
eigvals_ATA, eigvecs_ATA = torch.linalg.eigh(ATA)
# eigh 返回升序，反转成降序
eigvals_ATA = eigvals_ATA.flip(0)
eigvecs_ATA = eigvecs_ATA.flip(1)

print("\nA^T A =\n", ATA)
print("A^T A 的特征值（降序）:", eigvals_ATA.tolist())
print("奇异值的平方:", (S ** 2).tolist())
print("\nλ(A^T A) == σ²?", torch.allclose(eigvals_ATA[:len(S)], S**2, atol=1e-4))

# 右奇异向量 = A^T A 的特征向量
print("\n右奇异向量 V（列）:\n", Vh.T)
print("A^T A 的特征向量（列）:\n", eigvecs_ATA)
print("\n注意：特征向量的符号可能不同（v 和 -v 都是特征向量）")
# 比较绝对值
print("绝对值相等?", torch.allclose(Vh.T.abs(), eigvecs_ATA.abs(), atol=1e-4))

In [ ]:
# AA^T 的特征分解
AAT = A @ A.T
eigvals_AAT, eigvecs_AAT = torch.linalg.eigh(AAT)
eigvals_AAT = eigvals_AAT.flip(0)
eigvecs_AAT = eigvecs_AAT.flip(1)

print("AA^T =\n", AAT)
print("AA^T 的特征值（降序）:", eigvals_AAT.tolist())
print("奇异值的平方:", (S ** 2).tolist())
print("λ(AA^T) == σ²?", torch.allclose(eigvals_AAT[:len(S)], S**2, atol=1e-4))

# 左奇异向量 = AA^T 的特征向量
print("\n左奇异向量 U（列）:\n", U)
print("AA^T 的特征向量（列）:\n", eigvecs_AAT)
print("绝对值相等?", torch.allclose(U.abs(), eigvecs_AAT.abs(), atol=1e-4))

# 奇异值一定非负
print("\n所有奇异值 >= 0?", (S >= 0).all().item())
print("非零奇异值个数 = rank(A) =", (S > 1e-10).sum().item())
print("torch.linalg.matrix_rank(A) =", torch.linalg.matrix_rank(A).item())

## 7. 低秩近似与 Eckart-Young 定理

### 7.1 截断 SVD 与低秩近似

SVD 的最大应用之一是**低秩近似**（low-rank approximation）。

将 SVD 写成外积之和的形式：

$$
A = \sum_{i=1}^{r} \sigma_i u_i v_i^T
$$

其中 $u_i$ 是 U 的第 i 列，$v_i$ 是 V 的第 i 列。每个 $\sigma_i u_i v_i^T$ 是一个秩-1 矩阵。

**截断 SVD**：只保留前 k 个最大的奇异值，得到秩-k 近似：

$$
A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T = U_k \Sigma_k V_k^T
$$

### 7.2 Eckart-Young 定理

> **Eckart-Young-Mirsky 定理**：在所有秩不超过 k 的矩阵中，$A_k$（截断 SVD）是 A 的最佳近似，即：
> $$
> \|A - A_k\|_F = \min_{\text{rank}(B) \leq k} \|A - B\|_F
> $$
> 且误差为：$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \cdots + \sigma_r^2}$

这就是图像压缩、推荐系统、PCA 降维等众多应用的数学基础。

In [ ]:
# 低秩近似示例
A = torch.randn(10, 10)
U, S, Vh = torch.linalg.svd(A)

print("原始矩阵 A shape:", A.shape)
print("奇异值:", S.tolist())
print("\n奇异值平方（各秩-1分量的能量）:")
for i, s in enumerate(S):
    energy = s**2 / (S**2).sum() * 100
    print(f"  σ{i+1} = {s:.4f}, 能量占比 = {energy:.2f}%")

# 不同 k 的低秩近似
print("\n--- 低秩近似误差 ---")
print(f"{'k':>3} {'rank(Ak)':>8} {'||A-Ak||_F':>14} {'相对误差':>10} {'压缩率':>8}")
for k in [1, 2, 3, 5, 7, 10]:
    # 截断 SVD
    Uk = U[:, :k]
    Sk = S[:k]
    Vhk = Vh[:k, :]
    Ak = Uk @ torch.diag(Sk) @ Vhk
    
    error = torch.norm(A - Ak).item()
    rel_error = error / torch.norm(A).item() * 100
    # 压缩率：原始参数 100，Ak 参数 k*(10+10+1)
    orig_params = A.numel()
    ak_params = k * (A.shape[0] + A.shape[1] + 1)
    compression = (1 - ak_params / orig_params) * 100
    
    print(f"{k:>3} {torch.linalg.matrix_rank(Ak).item():>8} {error:>14.6f} {rel_error:>9.2f}% {compression:>7.1f}%")

In [ ]:
# 验证 Eckart-Young 定理：截断 SVD 是最佳秩-k 近似
A = torch.randn(8, 6)
U, S, Vh = torch.linalg.svd(A)
k = 3

# 截断 SVD 近似
Uk = U[:, :k]
Sk = S[:k]
Vhk = Vh[:k, :]
Ak_svd = Uk @ torch.diag(Sk) @ Vhk
error_svd = torch.norm(A - Ak_svd).item()

# 随机生成一个秩-k 矩阵，比较误差
best_random_error = float('inf')
for trial in range(100):
    B = torch.randn(8, k) @ torch.randn(k, 6)  # 随机秩-k 矩阵
    error = torch.norm(A - B).item()
    best_random_error = min(best_random_error, error)

print(f"k = {k}")
print(f"截断 SVD 误差 ||A-Ak||_F = {error_svd:.6f}")
print(f"100个随机秩-{k} 矩阵的最小误差 = {best_random_error:.6f}")
print(f"截断 SVD 误差 <= 随机误差? {error_svd <= best_random_error + 1e-5}")
print(f"\n理论误差 = sqrt(σ{k+1}²+...+σr²) = {torch.sqrt((S[k:]**2).sum()).item():.6f}")
print(f"实际误差 = {error_svd:.6f}")
print("匹配?", torch.isclose(torch.tensor(error_svd), torch.sqrt((S[k:]**2).sum()), atol=1e-4))

## 8. 应用

### 8.1 图像压缩（低秩近似）

图像可以表示为矩阵（灰度图）或三个矩阵（RGB）。用截断 SVD 保留前 k 个奇异值，可以在保留主要视觉信息的同时大幅压缩数据。

In [ ]:
# 生成一个测试图像（渐变 + 图案），模拟真实图像
size = 128
x = torch.linspace(0, 1, size)
y = torch.linspace(0, 1, size)
X, Y = torch.meshgrid(x, y, indexing='ij')
# 构造一个有结构的"图像"
image = (torch.sin(10 * X) * torch.cos(8 * Y) + 
         0.5 * torch.sin(20 * X + 5 * Y) + 
         0.3 * X + 0.2 * Y)
image = (image - image.min()) / (image.max() - image.min())  # 归一化到 [0,1]

print("图像矩阵 shape:", image.shape)
print("图像秩:", torch.linalg.matrix_rank(image).item())

# SVD
U, S, Vh = torch.linalg.svd(image)
print("\n奇异值（前20个）:", S[:20].tolist())
print("奇异值衰减比 σ1/σ20 =", (S[0] / S[19]).item())

# 不同 k 的压缩效果
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

axes[0].imshow(image.numpy(), cmap='gray')
axes[0].set_title(f"原始 (rank={torch.linalg.matrix_rank(image).item()})")
axes[0].axis('off')

for idx, k in enumerate([1, 5, 10, 20, 50]):
    Uk = U[:, :k]
    Sk = S[:k]
    Vhk = Vh[:k, :]
    compressed = Uk @ torch.diag(Sk) @ Vhk
    compressed = torch.clamp(compressed, 0, 1)
    
    error = torch.norm(image - compressed) / torch.norm(image) * 100
    orig_size = image.numel()
    compressed_size = k * (image.shape[0] + image.shape[1] + 1)
    compression = (1 - compressed_size / orig_size) * 100
    
    axes[idx+1].imshow(compressed.numpy(), cmap='gray')
    axes[idx+1].set_title(f"k={k}, 误差={error:.1f}%, 压缩={compression:.0f}%")
    axes[idx+1].axis('off')

plt.suptitle("SVD 图像压缩：截断奇异值", fontsize=14)
plt.tight_layout()
plt.show()

### 8.2 PCA 的 SVD 实现

主成分分析（PCA）通常通过协方差矩阵的特征分解实现，但更数值稳定的方式是直接对数据矩阵做 SVD。

对于中心化后的数据矩阵 $X \in \mathbb{R}^{n \times d}$（n 个样本，d 个特征）：

$$
X = U \Sigma V^T
$$

- **主成分方向** = V 的列（右奇异向量）
- **主成分方差** = $\sigma_i^2 / (n-1)$
- **投影到主成分** = $X V_k = U_k \Sigma_k$

SVD 方法避免了直接计算 $X^T X$（可能损失数值精度）。

In [ ]:
# PCA 的 SVD 实现 vs 协方差特征分解实现
torch.manual_seed(42)
n_samples = 500

# 生成有相关性的 3D 数据
z1 = torch.randn(n_samples)
z2 = torch.randn(n_samples)
X = torch.stack([
    z1 + 0.1 * torch.randn(n_samples),
    0.5 * z1 + z2 + 0.1 * torch.randn(n_samples),
    0.3 * z1 + 0.7 * z2 + 0.1 * torch.randn(n_samples)
], dim=1)

# 中心化
X_centered = X - X.mean(dim=0)

# 方法1：协方差矩阵特征分解
cov = (X_centered.T @ X_centered) / (n_samples - 1)
eigvals_cov, eigvecs_cov = torch.linalg.eigh(cov)
eigvals_cov = eigvals_cov.flip(0)  # 降序
eigvecs_cov = eigvecs_cov.flip(1)

# 方法2：SVD（更稳定）
U, S, Vh = torch.linalg.svd(X_centered, full_matrices=False)
# SVD 的奇异值平方 / (n-1) = 协方差特征值
eigvals_svd = S**2 / (n_samples - 1)
eigvecs_svd = Vh.T  # 右奇异向量

print("=== 协方差特征分解 ===")
print("特征值（方差解释）:", eigvals_cov.tolist())
print("方差解释比例:", (eigvals_cov / eigvals_cov.sum() * 100).tolist())

print("\n=== SVD 实现 ===")
print("奇异值:", S.tolist())
print("σ²/(n-1) = 特征值:", eigvals_svd.tolist())
print("方差解释比例:", (eigvals_svd / eigvals_svd.sum() * 100).tolist())

print("\n两种方法特征值一致?", torch.allclose(eigvals_cov, eigvals_svd, atol=1e-4))
print("特征向量绝对值一致?", torch.allclose(eigvecs_cov.abs(), eigvecs_svd.abs(), atol=1e-4))

# 投影到前 2 个主成分
k = 2
X_projected_svd = X_centered @ eigvecs_svd[:, :k]
print(f"\n投影到前 {k} 个主成分: {X_centered.shape} → {X_projected_svd.shape}")
print(f"保留方差比例: {eigvals_svd[:k].sum()/eigvals_svd.sum()*100:.2f}%")

In [ ]:
# 可视化 PCA 降维结果
fig = plt.figure(figsize=(14, 5))

# 原始 3D 数据
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_centered[:, 0], X_centered[:, 1], X_centered[:, 2],
            alpha=0.4, s=15, c='blue')
ax1.set_title("原始 3D 数据")
ax1.set_xlabel("x1"); ax1.set_ylabel("x2"); ax1.set_zlabel("x3")

# PCA 降维到 2D
ax2 = fig.add_subplot(122)
ax2.scatter(X_projected_svd[:, 0], X_projected_svd[:, 1],
            alpha=0.4, s=15, c='red')
ax2.set_title(f"PCA 降维到 2D (保留 {eigvals_svd[:2].sum()/eigvals_svd.sum()*100:.1f}% 方差)")
ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

### 8.3 伪逆（Moore-Penrose Pseudoinverse）

对于奇异矩阵或非方阵，逆矩阵不存在，但可以定义**伪逆** $A^+$：

$$
A^+ = V \Sigma^+ U^T
$$

其中 $\Sigma^+$ 是将 $\Sigma$ 的非零对角线元素取倒数，零元素保持为零。

伪逆的性质：
- $A A^+ A = A$
- $A^+ A A^+ = A^+$
- $(A A^+)^T = A A^+$
- $(A^+ A)^T = A^+ A$

**应用**：求解最小二乘问题（当 A 列满秩时，$x = A^+ b$ 就是最小二乘解）。

In [ ]:
# 伪逆：奇异矩阵和非方阵
# 例子1：奇异方阵
A_singular = torch.tensor([[1.0, 2.0],
                            [2.0, 4.0]])  # det=0
print("奇异矩阵 A =\n", A_singular)
print("det(A) =", torch.det(A_singular).item(), "(=0, 不可逆)")

# 伪逆
A_pinv = torch.linalg.pinv(A_singular)
print("\n伪逆 A+ =\n", A_pinv)

# 验证伪逆性质
print("\n验证伪逆性质:")
print("  A A+ A == A?", torch.allclose(A_singular @ A_pinv @ A_singular, A_singular, atol=1e-5))
print("  A+ A A+ == A+?", torch.allclose(A_pinv @ A_singular @ A_pinv, A_pinv, atol=1e-5))
print("  (A A+)^T == A A+?", torch.allclose((A_singular @ A_pinv).T, A_singular @ A_pinv, atol=1e-5))

# 用 SVD 手动计算伪逆
U, S, Vh = torch.linalg.svd(A_singular)
print("\nSVD 奇异值:", S.tolist())
S_pinv = torch.zeros_like(S)
S_pinv[S > 1e-10] = 1.0 / S[S > 1e-10]
A_pinv_svd = Vh.T @ torch.diag(S_pinv) @ U.T
print("SVD 计算的伪逆:\n", A_pinv_svd)
print("与 pinv 一致?", torch.allclose(A_pinv_svd, A_pinv, atol=1e-5))

In [ ]:
# 例子2：非方阵的伪逆与最小二乘
# 超定方程组：方程数 > 未知数，通常无解，求最小二乘解
A = torch.tensor([[1.0, 1.0],
                  [1.0, 2.0],
                  [1.0, 3.0],
                  [1.0, 4.0]])  # 4x2，超定
b = torch.tensor([1.0, 2.0, 3.0, 4.0])

print("超定系统 A (4x2), b (4):")
print("A =\n", A)
print("b =", b.tolist())

# 伪逆求解
x_pinv = torch.linalg.pinv(A) @ b
print("\n伪逆解 x = A+ b =", x_pinv.tolist())

# 与 linalg.lstsq 对比
x_lstsq, _, _, _ = torch.linalg.lstsq(A, b)
print("lstsq 解 x =", x_lstsq.tolist())
print("一致?", torch.allclose(x_pinv, x_lstsq, atol=1e-5))

# 验证：这是最小二乘解（残差最小）
residual = A @ x_pinv - b
print("\n残差 A@x - b =", residual.tolist())
print("残差平方和 =", (residual**2).sum().item())

# 可视化：线性拟合
fig, ax = plt.subplots(figsize=(7, 5))
x_data = A[:, 1].numpy()
y_data = b.numpy()
ax.scatter(x_data, y_data, c='blue', s=50, label='数据点')
x_line = torch.linspace(0, 5, 100)
y_line = x_pinv[0] + x_pinv[1] * x_line
ax.plot(x_line.numpy(), y_line.numpy(), 'r-', linewidth=2, 
        label=f'最小二乘拟合: y={x_pinv[0]:.2f}+{x_pinv[1]:.2f}x')
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("超定方程组的最小二乘解（伪逆）")
plt.tight_layout()
plt.show()

## 课后练习

### 基础题

1. 对矩阵 $A = \begin{pmatrix} 2 & 1 & 0 \\ 1 & 2 & 1 \\ 0 & 1 & 2 \end{pmatrix}$，分别进行 LU、QR、Cholesky 分解，验证每种分解的重构等式。

2. 用 LU 分解手动解方程组 $\begin{pmatrix} 2 & 1 \\ 4 & 3 \end{pmatrix} x = \begin{pmatrix} 3 \\ 7 \end{pmatrix}$，写出前向代入和后向代入的每一步。

3. 手动实现 Gram-Schmidt 正交化，对矩阵 $A = \begin{pmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{pmatrix}$ 求 Q 和 R，与 `torch.linalg.qr` 对比。

### SVD 专题

4. 对一个 $5 \times 4$ 的随机矩阵做 SVD，验证：
   - $A = U\Sigma V^T$
   - $U^T U = I$，$V^T V = I$
   - $A^T A$ 的特征值 = 奇异值平方
   - 非零奇异值个数 = rank(A)

5. 对一个 $100 \times 100$ 的随机矩阵，画出奇异值的折线图和累积能量占比图，确定保留 95% 能量需要多少个奇异值。

6. 验证 Eckart-Young 定理：对一个 $8 \times 6$ 矩阵，取 k=2，证明截断 SVD 的误差 $\|A-A_k\|_F = \sqrt{\sigma_3^2+\cdots+\sigma_r^2}$，且小于任意随机秩-2 矩阵的误差。

### 应用题

7. 图像压缩：生成一个 $256 \times 256$ 的测试图像，分别用 k=5, 10, 20, 50, 100 做截断 SVD 压缩，计算每种的 PSNR（峰值信噪比）和压缩率，画出压缩率-PSNR 曲线。

8. PCA 的 SVD 实现：生成一个 4 维相关数据集（200 个样本），用 SVD 做 PCA 降维到 2 维，计算每个主成分的方差解释比例，可视化降维结果。

9. 伪逆与最小二乘：生成一个超定系统（50 个样本，3 个特征），分别用 `pinv`、`lstsq`、和正规方程 $(A^T A)^{-1} A^T b$ 三种方法求解最小二乘解，对比结果和数值稳定性（当 A 接近列秩亏时）。

### 综合题

10. 比较四种分解的适用条件和计算复杂度：
    - LU：适用条件？复杂度？
    - QR：适用条件？复杂度？
    - Cholesky：适用条件？复杂度？
    - SVD：适用条件？复杂度？
    用一个表格总结，并各举一个典型应用场景。

11. 用 SVD 证明：对于任意矩阵 A，$\|A\|_2 = \sigma_{\max}(A)$（谱范数 = 最大奇异值），其中 $\|A\|_2 = \max_{\|x\|_2=1} \|Ax\|_2$。用代码验证一个 $4 \times 3$ 矩阵。